# PhishGuard – אימון מודל BERT לזיהוי פישינג

## הוראות שימוש

1. **הפעל מעבד GPU**: Runtime → Change runtime type → Hardware accelerator → **T4 GPU** → Save
2. הרץ את התאים לפי הסדר, מלמעלה למטה
3. בתא 3 תצתרך **GitHub Personal Access Token** (הריפו פרטי):
   - GitHub → Settings → Developer settings → Personal access tokens → Tokens (classic) → Generate new token → סקופ `repo`
4. **מרכיב**: לפני הרצה, העלה ל-Google Drive שלך תיקייה בשם `PhishGuardData` ובתוכה את הקבצים האמיתיים שלך:
   - `emails.csv`
   - `enron_legitimate.csv`
   - `phishtank.csv`
5. בסוף האימון, הקובץ `best_model.pt` יישמר ב-Drive שלך – תוריד אותו ושים במחשב המקומי ב-`backend/ML/checkpoints/best_model.pt`

In [ ]:
# תא 1 – בדיקת GPU
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ אין GPU – לך ל-Runtime > Change runtime type > T4 GPU והרץ מחדש")

In [ ]:
# תא 2 – שכפול את הרצון (הרשם המלא ב-GitHub)
from getpass import getpass

GITHUB_USER = "lynnnagel"
REPO = "FinalProject2026"
BRANCH = "develop"  # שנה לענף הרלוונטי אם צריך

token = getpass("הדבק GitHub Personal Access Token: ")
!git clone -b {BRANCH} https://{token}@github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}/backend

In [ ]:
# תא 3 – התקנת חבילות חסרות (torch ו-pandas כבר מותקנות ב-Colab)
!pip install -q transformers==4.46.0

## נתונים
הקבצים בתוך `backend/ML/data/` ברשם היו קבצי Git-LFS placeholder (לא הנתונים האמיתיים). התא הבא מעתיק את הקבצים האמיתיים מ-Google Drive שלך (תיקייה `MyDrive/PhishGuardData/`).

In [ ]:
# תא 4 – חיבור Google Drive והעתקת הקבצים האמיתיים לתיקיית הנתונים
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_SRC = "/content/drive/MyDrive/PhishGuardData"
DATA_DST = "ML/data"
os.makedirs(DATA_DST, exist_ok=True)

for fname in ["emails.csv", "enron_legitimate.csv", "phishtank.csv"]:
    src = os.path.join(DATA_SRC, fname)
    if os.path.exists(src):
        !cp "{src}" "{DATA_DST}/"
        print(f"✓ {fname} הועבר")
    else:
        print(f"✗ {fname} לא נמצא ב-{DATA_SRC} – שים אותו שם והרץ תא זו מחדש")

In [ ]:
# תא 5 – הכנת נתונים (ניקוי, זיהוי עמודות, פיצול 70/15/15)
!python ML/prepare_data.py --data_dir ML/data --output_dir ML/data/processed

In [ ]:
# תא 6 – אימון המודל (כל 3 אפוכים, על גבי T4 לקח בדרך כלשעה-שעתיים)
!python ML/train.py --data_dir ML/data/processed --output_dir ML/checkpoints --epochs 3 --batch_size 16

In [ ]:
# תא 7 – שמירת ה-checkpoint ל-Drive כדי שלא ייעלם עם סגירת ה-Colab
import shutil, os
os.makedirs("/content/drive/MyDrive/PhishGuardData/checkpoints", exist_ok=True)
shutil.copy("ML/checkpoints/best_model.pt", "/content/drive/MyDrive/PhishGuardData/checkpoints/best_model.pt")
print("נשמר ב-Drive: PhishGuardData/checkpoints/best_model.pt")

## שלב אחרון – החזרה לפרויקט המקומי

1. מ-Drive: הורידי את `PhishGuardData/checkpoints/best_model.pt`
2. העתיקי אותו במקום `backend/ML/checkpoints/best_model.pt` (מחליף את הקובץ הריק) בפרויקט המקומי שלך
3. `git add backend/ML/checkpoints/best_model.pt` → `git commit` → `git push`
4. כשתפעילי את השרת (`backend/server.py`), `bert_model.py` יטען אותו אוטומטית והמערכת תתחיל להשתמש באנסמבל BERT+היורסטיקה

**מערכת**: אם הקובץ כבד (מאות מייבייטים), ו-`.gitignore` מכיל את `*.pt` – תצטרכי להסיר את ההתעלמה הזו או להוסיף בפועל -f.